# 数学基础

机器学习涉及的数学基础主要来自四个领域：

| 领域 | 核心用途 |
|------|----------|
| **线性代数** | 数据表示（矩阵/向量）、模型参数、PCA、SVD |
| **微积分** | 损失函数求导、梯度下降、反向传播 |
| **概率与统计** | 模型假设、最大似然、贝叶斯方法 |
| **优化理论** | 参数学习、凸优化、约束优化（SVM）|

# 一、线性代数

## 1.1 向量与矩阵

在机器学习中：
- **一个样本** = 一个向量 $x \in \mathbb{R}^n$（$n$ 个特征）
- **数据集** = 矩阵 $X \in \mathbb{R}^{m \times n}$（$m$ 个样本，$n$ 个特征）
- **模型参数** = 权重向量 $w \in \mathbb{R}^n$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 向量基本运算
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("向量加法:   ", a + b)
print("数乘:       ", 2 * a)
print("点积 a·b:   ", np.dot(a, b))          # = 1*4 + 2*5 + 3*6 = 32
print("a 的模长:   ", np.linalg.norm(a))      # ||a|| = sqrt(1+4+9)

# 矩阵运算
A = np.array([[1, 2], [3, 4], [5, 6]])  # (3,2)
B = np.array([[7, 8], [9, 10]])          # (2,2)
print("\nA shape:", A.shape)
print("A @ B =\n", A @ B)               # 矩阵乘法 (3,2)@(2,2)=(3,2)
print("A.T =\n",   A.T)                 # 转置 (2,3)

## 1.2 范数（Norm）

范数衡量向量的"大小"，在正则化中直接使用。

$$\|x\|_1 = \sum_i |x_i| \quad \text{（L1，菱形）}$$

$$\|x\|_2 = \sqrt{\sum_i x_i^2} \quad \text{（L2，球形，最常用）}$$

$$\|x\|_p = \left(\sum_i |x_i|^p\right)^{1/p}$$

- **L1 正则化**：使权重稀疏（部分权重精确为 0），自动特征选择
- **L2 正则化**：使权重均匀缩小，不产生稀疏性

In [ ]:
x = np.array([3.0, -4.0])
print(f"L1 范数: {np.linalg.norm(x, 1):.2f}")   # 3+4=7
print(f"L2 范数: {np.linalg.norm(x, 2):.2f}")   # sqrt(9+16)=5
print(f"L∞ 范数: {np.linalg.norm(x, np.inf):.2f}")  # max(3,4)=4

# 可视化 L1 与 L2 的单位球（二维）
theta = np.linspace(0, 2 * np.pi, 300)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, p, title in zip(axes, [1, 2], ['L1 单位球（菱形）', 'L2 单位球（圆形）']):
    pts = []
    for t in theta:
        v = np.array([np.cos(t), np.sin(t)])
        v = v / np.linalg.norm(v, p)
        pts.append(v)
    pts = np.array(pts)
    ax.plot(pts[:, 0], pts[:, 1], 'steelblue', linewidth=2)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_aspect('equal')
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 1.3 特征值与特征向量

若 $Av = \lambda v$（$v \neq 0$），则 $\lambda$ 是**特征值**，$v$ 是对应的**特征向量**。

直觉：矩阵 $A$ 作用在特征向量 $v$ 上，只是把 $v$ 拉伸/压缩了 $\lambda$ 倍，方向不变。

**在 ML 中的应用：**
- PCA 的主成分 = 协方差矩阵的特征向量（按特征值降序排列）
- 特征值表示该方向的数据方差大小

In [ ]:
np.random.seed(42)
# 构造一个协方差矩阵（对称正定）
data = np.random.randn(200, 2) @ np.array([[2, 1], [0.5, 1]])
cov  = np.cov(data.T)
eigenvalues, eigenvectors = np.linalg.eig(cov)

# 按特征值降序排列
order = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

print("协方差矩阵:\n", cov.round(3))
print("\n特征值:", eigenvalues.round(3))
print("特征向量（列）:\n", eigenvectors.round(3))

# 可视化：特征向量方向即主成分方向
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(data[:, 0], data[:, 1], alpha=0.4, s=15, color='steelblue')
mean = data.mean(axis=0)
for i, (val, vec) in enumerate(zip(eigenvalues, eigenvectors.T)):
    ax.annotate('', mean + np.sqrt(val) * vec, mean,
                arrowprops=dict(arrowstyle='->', color='tomato', lw=2))
    ax.text(*(mean + np.sqrt(val) * vec * 1.1), f'PC{i+1} λ={val:.2f}',
            fontsize=9, color='tomato')
ax.set_title('协方差矩阵的特征向量 = PCA 主成分方向')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 1.4 奇异值分解（SVD）

任意矩阵 $A \in \mathbb{R}^{m \times n}$ 都可以分解为：

$$A = U \Sigma V^T$$

- $U \in \mathbb{R}^{m \times m}$：左奇异向量（正交矩阵）
- $\Sigma \in \mathbb{R}^{m \times n}$：对角矩阵，对角元素为奇异值 $\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$
- $V \in \mathbb{R}^{n \times n}$：右奇异向量（正交矩阵）

**在 ML 中的应用：**
- PCA 的底层实现（对数据矩阵做 SVD 等价于对协方差矩阵做特征分解）
- 矩阵压缩：保留前 $k$ 个奇异值，低秩近似
- 推荐系统中的矩阵分解

In [ ]:
from sklearn.datasets import load_sample_image

# 用灰度图像演示 SVD 压缩
img = load_sample_image('china.jpg')
gray = img.mean(axis=2)   # 转灰度 (427, 640)

U, S, Vt = np.linalg.svd(gray, full_matrices=False)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, k in zip(axes, [5, 20, 50, len(S)]):
    # 取前 k 个奇异值重建
    recon = (U[:, :k] * S[:k]) @ Vt[:k, :]
    ax.imshow(recon, cmap='gray')
    ratio = k * (U.shape[0] + Vt.shape[1] + 1) / gray.size * 100
    ax.set_title(f'k={k}  压缩率={ratio:.1f}%')
    ax.axis('off')

plt.suptitle('SVD 图像压缩：k 越大还原越清晰', fontsize=13)
plt.tight_layout()
plt.show()

# 奇异值衰减曲线
plt.figure(figsize=(7, 3))
plt.plot(S[:80], 'steelblue', linewidth=2)
plt.xlabel('奇异值序号')
plt.ylabel('奇异值大小')
plt.title('奇异值快速衰减——前几个奇异值包含大部分信息')
plt.tight_layout()
plt.show()

# 二、微积分

## 2.1 导数与偏导数

**导数**：函数在某点的瞬时变化率

$$f'(x) = \lim_{\Delta x \to 0} \frac{f(x + \Delta x) - f(x)}{\Delta x}$$

**偏导数**：多元函数对某一变量的导数（其余变量视为常数）

$$\frac{\partial f}{\partial x_1} = \lim_{\Delta x_1 \to 0} \frac{f(x_1 + \Delta x_1, x_2, \ldots) - f(x_1, x_2, \ldots)}{\Delta x_1}$$

**梯度**：所有偏导数组成的向量，指向函数值上升最快的方向

$$\nabla f(x) = \left(\frac{\partial f}{\partial x_1}, \frac{\partial f}{\partial x_2}, \ldots, \frac{\partial f}{\partial x_n}\right)^T$$

In [ ]:
# 梯度场可视化：f(x,y) = x^2 + 2y^2
x = np.linspace(-3, 3, 30)
y = np.linspace(-3, 3, 30)
X, Y = np.meshgrid(x, y)
Z = X**2 + 2*Y**2          # 函数值

# 梯度 = (2x, 4y)
dZdx = 2 * X
dZdy = 4 * Y

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 等高线图 + 梯度箭头
ax1.contourf(X, Y, Z, levels=20, cmap='coolwarm', alpha=0.7)
ax1.contour(X, Y, Z, levels=20, colors='gray', linewidths=0.5)
ax1.quiver(X[::2, ::2], Y[::2, ::2],
           dZdx[::2, ::2], dZdy[::2, ::2],
           color='black', alpha=0.6, scale=80)
ax1.set_title('f(x,y)=x²+2y² 的梯度场\n（箭头 = 上升最快方向）')

# 梯度下降轨迹
pt = np.array([2.5, 2.5])
path = [pt.copy()]
for _ in range(30):
    grad = np.array([2 * pt[0], 4 * pt[1]])
    pt = pt - 0.1 * grad
    path.append(pt.copy())
path = np.array(path)
ax2.contourf(X, Y, Z, levels=20, cmap='coolwarm', alpha=0.7)
ax2.plot(path[:, 0], path[:, 1], 'ko-', markersize=4, linewidth=1.5, label='梯度下降路径')
ax2.scatter([0], [0], c='red', s=100, zorder=5, label='最小值(0,0)')
ax2.set_title('梯度下降：沿梯度反方向迭代')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 2.2 链式法则（Chain Rule）

神经网络反向传播的核心数学工具。

若 $z = f(g(x))$，则：

$$\frac{dz}{dx} = \frac{dz}{dg} \cdot \frac{dg}{dx}$$

**多元链式法则**：若 $z = f(u, v)$，$u = g(x)$，$v = h(x)$：

$$\frac{dz}{dx} = \frac{\partial z}{\partial u}\frac{du}{dx} + \frac{\partial z}{\partial v}\frac{dv}{dx}$$

**应用举例（逻辑回归的损失梯度）：**

$$L = -[y \log \sigma(z) + (1-y)\log(1-\sigma(z))], \quad z = w^T x + b$$

$$\frac{\partial L}{\partial w} = (\sigma(z) - y) \cdot x \quad \text{（链式法则展开后的简洁形式）}$$

## 2.3 泰勒展开

将函数在某点 $x_0$ 附近用多项式近似：

$$f(x) \approx f(x_0) + f'(x_0)(x - x_0) + \frac{f''(x_0)}{2!}(x-x_0)^2 + \cdots$$

**一阶近似（线性近似）** 是梯度下降的理论依据：

$$f(x_0 + \Delta x) \approx f(x_0) + \nabla f(x_0)^T \Delta x$$

选 $\Delta x = -\eta \nabla f(x_0)$ 时，$f$ 减小 $\eta \|\nabla f\|^2$，这就是梯度下降每步"下坡"的保证。

**二阶近似（牛顿法）** 利用 Hessian 矩阵（二阶导数矩阵）：

$$f(x_0 + \Delta x) \approx f(x_0) + \nabla f^T \Delta x + \frac{1}{2} \Delta x^T H \Delta x$$

令梯度为零得 $\Delta x = -H^{-1} \nabla f$，收敛比梯度下降快，但计算 $H^{-1}$ 代价大。

In [ ]:
# 泰勒展开可视化：sin(x) 在 x=0 处的逐阶近似
x_plot = np.linspace(-np.pi, np.pi, 300)
f_true = np.sin(x_plot)

# 泰勒展开各阶项（在 x0=0 展开）
orders = {
    '1阶: x':                  x_plot,
    '3阶: x - x³/6':           x_plot - x_plot**3 / 6,
    '5阶: +x⁵/120':            x_plot - x_plot**3/6 + x_plot**5/120,
    '7阶: -x⁷/5040':           x_plot - x_plot**3/6 + x_plot**5/120 - x_plot**7/5040,
}

plt.figure(figsize=(9, 4))
plt.plot(x_plot, f_true, 'k-', linewidth=3, label='sin(x) 真实值')
for label, approx in orders.items():
    plt.plot(x_plot, approx, '--', linewidth=1.5, label=label)
plt.ylim(-2, 2)
plt.legend(fontsize=8, loc='upper right')
plt.title('sin(x) 的泰勒展开：阶数越高，有效范围越大')
plt.tight_layout()
plt.show()

# 三、概率与统计

## 3.1 概率基础

**条件概率**：在事件 $B$ 已发生的前提下，$A$ 发生的概率

$$P(A|B) = \frac{P(A \cap B)}{P(B)}$$

**贝叶斯定理**：ML 中用于从观测数据推断参数的核心公式

$$P(\theta | D) = \frac{P(D | \theta) \cdot P(\theta)}{P(D)}$$

- $P(\theta)$：先验概率（对参数的初始认知）
- $P(D|\theta)$：似然（参数为 $\theta$ 时观测到数据 $D$ 的概率）
- $P(\theta|D)$：后验概率（看到数据后对参数的更新认知）
- $P(D)$：证据（归一化常数）

**全概率公式**：

$$P(B) = \sum_i P(B|A_i) P(A_i)$$

## 3.2 常用概率分布

| 分布 | 参数 | 场景 |
|------|------|------|
| 伯努利 | $p$ | 单次抛硬币（0/1） |
| 二项分布 | $n, p$ | $n$ 次独立试验中成功次数 |
| 正态分布 | $\mu, \sigma^2$ | 自然界中最广泛，线性回归误差假设 |
| 均匀分布 | $a, b$ | 等可能随机初始化 |
| 指数分布 | $\lambda$ | 等待时间建模 |

In [ ]:
from scipy import stats

x_cont = np.linspace(-5, 5, 300)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 正态分布：不同均值和方差
for mu, sigma, label in [(0,1,'μ=0,σ=1'), (0,2,'μ=0,σ=2'), (2,1,'μ=2,σ=1')]:
    axes[0].plot(x_cont, stats.norm.pdf(x_cont, mu, sigma), linewidth=2, label=label)
axes[0].set_title('正态分布 N(μ,σ²)')
axes[0].legend(fontsize=8)

# 中心极限定理：样本均值趋向正态
np.random.seed(0)
sample_means = [np.random.uniform(0,1,n).mean() for n in [1,5,30,100] for _ in range(2000)]
sample_means = np.array(sample_means).reshape(4, 2000)
for i, n in enumerate([1, 5, 30, 100]):
    axes[1].hist(sample_means[i], bins=40, alpha=0.6, density=True, label=f'n={n}')
axes[1].set_title('中心极限定理：样本量越大均值越正态')
axes[1].legend(fontsize=8)

# 指数分布
for lam, label in [(0.5,'λ=0.5'), (1,'λ=1'), (2,'λ=2')]:
    axes[2].plot(x_cont[x_cont>=0], stats.expon.pdf(x_cont[x_cont>=0], scale=1/lam),
                 linewidth=2, label=label)
axes[2].set_title('指数分布 Exp(λ)')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 3.3 期望、方差与协方差

$$\mathbb{E}[X] = \sum_x x \cdot P(X=x) \quad \text{（离散）} \quad \text{或} \quad \int x f(x) dx \quad \text{（连续）}$$

$$\text{Var}(X) = \mathbb{E}[(X - \mathbb{E}[X])^2] = \mathbb{E}[X^2] - (\mathbb{E}[X])^2$$

$$\text{Cov}(X, Y) = \mathbb{E}[(X - \mu_X)(Y - \mu_Y)]$$

**皮尔逊相关系数**（标准化的协方差）：

$$\rho_{XY} = \frac{\text{Cov}(X,Y)}{\sigma_X \sigma_Y} \in [-1, 1]$$

- $\rho = 1$：完全正线性相关；$\rho = -1$：完全负相关；$\rho = 0$：线性无关

In [ ]:
np.random.seed(42)
n = 500
x1 = np.random.randn(n)
x2_pos = x1 + 0.5 * np.random.randn(n)   # 正相关
x2_neg = -x1 + 0.5 * np.random.randn(n)  # 负相关
x2_unc = np.random.randn(n)               # 不相关

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, x2, title in zip(axes,
                          [x2_pos, x2_neg, x2_unc],
                          ['正相关', '负相关', '不相关']):
    rho = np.corrcoef(x1, x2)[0, 1]
    ax.scatter(x1, x2, alpha=0.3, s=10, color='steelblue')
    ax.set_title(f'{title}  ρ={rho:.2f}')
    ax.set_xlabel('X'); ax.set_ylabel('Y')

plt.suptitle('相关系数的直观含义', fontsize=13)
plt.tight_layout()
plt.show()

## 3.4 最大似然估计（MLE）

**核心思想**：找到参数 $\theta$，使得观测到当前数据的概率最大。

$$\hat{\theta}_{MLE} = \arg\max_\theta \prod_{i=1}^m P(x_i | \theta) = \arg\max_\theta \sum_{i=1}^m \log P(x_i | \theta)$$

（取对数将连乘转为求和，便于求导，且不改变最大值位置）

**应用举例：**
- 线性回归 = 假设误差服从正态分布时的 MLE → 等价于最小化 MSE
- 逻辑回归 = 假设标签服从伯努利分布时的 MLE → 等价于最小化交叉熵

In [ ]:
# MLE 估计正态分布参数
np.random.seed(0)
true_mu, true_sigma = 3.0, 1.5
samples = np.random.normal(true_mu, true_sigma, 200)

mu_mle    = samples.mean()
sigma_mle = samples.std()

# 扫描 mu 的对数似然曲线
mu_range = np.linspace(1, 5, 200)
log_lik  = [np.sum(stats.norm.logpdf(samples, mu, sigma_mle)) for mu in mu_range]

plt.figure(figsize=(8, 4))
plt.plot(mu_range, log_lik, 'steelblue', linewidth=2)
plt.axvline(mu_mle,  color='red',  linestyle='--', label=f'MLE μ̂={mu_mle:.3f}')
plt.axvline(true_mu, color='gray', linestyle=':',  label=f'真实 μ={true_mu}')
plt.xlabel('μ 的候选值')
plt.ylabel('对数似然 Σlog P(xᵢ|μ)')
plt.title('MLE：使对数似然最大的 μ 即为估计值')
plt.legend()
plt.tight_layout()
plt.show()
print(f"真实参数: μ={true_mu}, σ={true_sigma}")
print(f"MLE 估计: μ={mu_mle:.4f}, σ={sigma_mle:.4f}")

# 四、优化理论

## 4.1 凸函数与全局最优

**凸函数**：连线段始终在函数图像上方

$$f(\lambda x + (1-\lambda)y) \leq \lambda f(x) + (1-\lambda)f(y), \quad \forall \lambda \in [0,1]$$

**重要性质：凸函数的局部最小值 = 全局最小值**

- MSE（均方误差）是凸函数 → 线性回归梯度下降保证找到全局最优
- 交叉熵损失是凸函数 → 逻辑回归同理
- 神经网络的损失**非凸** → 只能找局部最优，但实践中效果仍好

**判断凸函数：Hessian 矩阵半正定**（$H \succeq 0$，即所有特征值 $\geq 0$）

In [ ]:
x_plot = np.linspace(-3, 3, 300)

# 凸函数 vs 非凸函数
f_convex    = x_plot**2                               # 凸
f_nonconvex = np.sin(2*x_plot) + 0.3*x_plot**2       # 非凸

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(x_plot, f_convex, 'steelblue', linewidth=2)
# 验证凸性：连接两点的线段在函数上方
p1, p2 = np.array([-2, 4]), np.array([2, 4])
ax1.plot([p1[0], p2[0]], [p1[1], p2[1]], 'r--', linewidth=2, label='弦（在曲线上方）')
ax1.scatter([0], [0], c='red', s=100, zorder=5, label='唯一全局最小值')
ax1.set_title('凸函数 f(x)=x²')
ax1.legend(fontsize=8)

ax2.plot(x_plot, f_nonconvex, 'steelblue', linewidth=2)
local_min_x = [-1.8, 0.7]
for xm in local_min_x:
    ax2.scatter([xm], [np.sin(2*xm) + 0.3*xm**2], c='orange', s=100, zorder=5)
ax2.annotate('局部最小', xy=(-1.8, np.sin(-3.6)+0.3*3.24), fontsize=9, color='orange')
ax2.annotate('全局最小', xy=(0.7,  np.sin(1.4)+0.3*0.49),  fontsize=9, color='red')
ax2.set_title('非凸函数：存在多个局部最小值')

plt.suptitle('凸 vs 非凸：梯度下降能否保证全局最优', fontsize=12)
plt.tight_layout()
plt.show()

## 4.2 梯度下降的三种变体

$$w \leftarrow w - \eta \nabla_w J(w)$$

| 方法 | 每步用多少数据 | 更新次数/轮 | 特点 |
|------|--------------|------------|------|
| **批量梯度下降（BGD）** | 全部 $m$ 个 | 1 | 稳定但慢，内存大 |
| **随机梯度下降（SGD）** | 1 个 | $m$ | 快但噪声大，可逃出局部最优 |
| **小批量梯度下降（MBGD）** | $b$ 个（batch） | $m/b$ | 兼顾两者，实践最常用 |

In [ ]:
# 学习率对收敛的影响
def f(w):  return w**2 + 2*w + 1       # f(w) = (w+1)^2，最小值在 w=-1
def grad(w): return 2*w + 2

w_init = 3.0
lrs = {'过小 η=0.01': 0.01, '适中 η=0.3': 0.3, '过大 η=1.1': 1.1}
colors = ['steelblue', 'seagreen', 'tomato']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

w_range = np.linspace(-4, 4, 200)
axes[0].plot(w_range, f(w_range), 'k-', linewidth=2)

for (name, lr), color in zip(lrs.items(), colors):
    w = w_init
    path = [w]
    for _ in range(25):
        w = w - lr * grad(w)
        path.append(w)
        if abs(w) > 10:
            break
    axes[0].plot([p for p in path], [f(p) for p in path],
                 'o-', color=color, markersize=4, linewidth=1.5, label=name)
axes[0].set_title('学习率对梯度下降的影响')
axes[0].legend(fontsize=8)
axes[0].set_ylim(-0.5, 20)

# 三种变体的损失曲线对比（模拟）
np.random.seed(0)
epochs = 50
bgd_loss  = 5 * np.exp(-0.15 * np.arange(epochs)) + 0.02
sgd_loss  = 5 * np.exp(-0.15 * np.arange(epochs)) + 0.3 * np.random.randn(epochs) + 0.02
mbgd_loss = 5 * np.exp(-0.15 * np.arange(epochs)) + 0.08 * np.random.randn(epochs) + 0.02

for loss, name, color in [(bgd_loss,'BGD','steelblue'), (sgd_loss,'SGD','tomato'), (mbgd_loss,'MBGD','seagreen')]:
    axes[1].plot(loss, color=color, linewidth=2, label=name)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('BGD / SGD / MBGD 收敛曲线对比（示意）')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4.3 拉格朗日乘子法（约束优化）

解决带等式约束的优化问题，SVM 的推导核心。

**问题形式：**

$$\min_x f(x) \quad \text{s.t.} \quad g(x) = 0$$

**构造拉格朗日函数：**

$$\mathcal{L}(x, \lambda) = f(x) + \lambda g(x)$$

最优解满足：$\nabla_x \mathcal{L} = 0$ 且 $g(x) = 0$

**KKT 条件**（不等式约束 $g(x) \leq 0$）：

$$\nabla_x f + \lambda \nabla_x g = 0, \quad \lambda \geq 0, \quad g(x) \leq 0, \quad \lambda g(x) = 0$$

最后一条 $\lambda g(x) = 0$ 称为**互补松弛条件**：要么约束不紧（$g < 0$，$\lambda = 0$），要么约束恰好取等（$g = 0$，$\lambda > 0$）。

In [ ]:
# 拉格朗日乘子法可视化：min x1^2 + x2^2  s.t. x1 + x2 = 1
x1 = np.linspace(-0.5, 1.5, 300)
x2_range = np.linspace(-0.5, 1.5, 300)
X1, X2 = np.meshgrid(x1, x2_range)
Z = X1**2 + X2**2   # 目标函数

fig, ax = plt.subplots(figsize=(6, 5))
cp = ax.contourf(X1, X2, Z, levels=20, cmap='coolwarm', alpha=0.7)
ax.contour(X1, X2, Z, levels=20, colors='gray', linewidths=0.5)

# 约束：x1 + x2 = 1
ax.plot(x1, 1 - x1, 'k-', linewidth=2, label='约束 x₁+x₂=1')

# 最优解（在约束线上目标函数最小）= (0.5, 0.5)
ax.scatter([0.5], [0.5], c='red', s=150, zorder=5, label='最优解 (0.5, 0.5)')

# 梯度在最优解处平行（拉格朗日条件）
ax.annotate('', (0.5 + 0.25, 0.5 + 0.25), (0.5, 0.5),
            arrowprops=dict(color='blue', arrowstyle='->', lw=2))
ax.annotate('', (0.5 + 0.18, 0.5 - 0.18), (0.5, 0.5),
            arrowprops=dict(color='green', arrowstyle='->', lw=2))
ax.text(0.78, 0.78, '∇f', color='blue', fontsize=10)
ax.text(0.7,  0.3,  '∇g', color='green', fontsize=10)

ax.set_title('拉格朗日乘子法：最优解处 ∇f ∥ ∇g')
ax.legend(fontsize=9)
plt.colorbar(cp, ax=ax)
plt.tight_layout()
plt.show()
print("分析：最优解 (0.5, 0.5) 处，∇f=(1,1) 与 ∇g=(1,1) 平行，λ=-1")

# 总结：数学工具与 ML 算法的对应关系

| 数学工具 | 对应 ML 算法/概念 |
|----------|-----------------|
| 矩阵乘法 $Xw$ | 线性回归、神经网络前向传播 |
| 特征值分解 | PCA、谱聚类 |
| SVD | PCA 实现、推荐系统矩阵分解 |
| L1/L2 范数 | Lasso/Ridge 正则化 |
| 偏导数 + 链式法则 | 反向传播（BP）、梯度下降 |
| 泰勒展开（一阶） | 梯度下降理论基础 |
| 泰勒展开（二阶）| 牛顿法、L-BFGS |
| 正态分布 | 线性回归误差假设 |
| 最大似然估计 | 所有参数学习的统一框架 |
| 贝叶斯定理 | 朴素贝叶斯、贝叶斯正则化（MAP） |
| 凸函数 | 确保梯度下降找到全局最优 |
| 拉格朗日乘子法 | SVM 最大间隔推导、约束优化 |